In [2]:
import os
import shutil
import random
import json
import subprocess

current_base_dir = os.getcwd()
raw_files_root = os.path.join(current_base_dir, "Raw_Files")
dp_models_root = os.path.join(current_base_dir, "DP_Models")

if not os.path.exists(raw_files_root) or not os.path.exists(dp_models_root):
    print("Error: Make sure 'Raw_Files' and 'DP_Models' directories exist here.")
    exit(1)

unum = input("Enter your username (for SLURM mail): ").strip()
graph_num = int(input("Enter the number of DP_models to generate per Water system: "))

seed_values = [1, 6534653463541, 652465462461] # Hardcoded
nsym_list = ["H", "O"] #This can become a user input

bash_script_content = r'''#!/bin/bash
nline_per_set=2000
[[ $# -ge 1 ]] && nline_per_set=$1
rm -fr set*
if ! [ -f box.raw ]; then
    echo "Error: box.raw not found. Exiting."
    exit 1
fi
split box.raw -l $nline_per_set -d -a 3 box.raw.
split coord.raw -l $nline_per_set -d -a 3 coord.raw.
for f in energy force virial atom_ener fparam atomic_dipole atomic_polarizability; do
    test -f ${f}.raw && split ${f}.raw -l $nline_per_set -d -a 3 ${f}.raw.
done
nset=$(ls -1 box.raw.* | wc -l)
nset_1=$((nset - 1))
for ii in $(seq 0 $nset_1); do
    pi=$(printf "%03d" $ii)
    mkdir -p "set.$pi"
    mv "box.raw.$pi" "set.$pi/box.raw"
    mv "coord.raw.$pi" "set.$pi/coord.raw"
    for f in energy force virial atom_ener fparam atomic_dipole atomic_polarizability; do
        test -f "${f}.raw.$pi" && mv "${f}.raw.$pi" "set.$pi/${f}.raw"
    done
    cd "set.$pi"
    python3 <<END
import os
import numpy as np
def c(f, n):
    if os.path.isfile(f):
        try:
            d = np.loadtxt(f, ndmin=2).astype(np.float32)
            np.save(n, d)
        except: pass
c('box.raw','box'); c('coord.raw','coord'); c('energy.raw','energy');
c('force.raw','force'); c('virial.raw','virial'); c('atom_ener.raw','atom_ener');
c('fparam.raw','fparam'); c('dipole.raw','dipole'); c('polarizability.raw','polarizability');
c('atomic_dipole.raw','atomic_dipole'); c('atomic_polarizability.raw','atomic_polarizability');
END
    rm -f *.raw
    cd ..
done
'''

water_systems = [d for d in os.listdir(raw_files_root) if os.path.isdir(os.path.join(raw_files_root, d)) and d.endswith("_Water")]
generated_data_paths = []

print("\n--- Processing WATER Systems with Relative Paths ---")

for system in water_systems:
    train_path = os.path.join(raw_files_root, system)
    
    print(f"\nProcessing Track: {system}")
    
    if os.path.isfile(os.path.join(train_path, "box.raw")) and os.path.isfile(os.path.join(train_path, "coord.raw")):
        test_path = os.path.join(train_path, 'Test')
        os.makedirs(test_path, exist_ok=True)
        
        raw_filenames = ['coord.raw', 'energy.raw', 'force.raw', 'box.raw', 'virial.raw']
        all_data = {}
        num_configurations = 0
        
        for filename in raw_filenames:
            filepath = os.path.join(train_path, filename)
            if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
                with open(filepath, 'r') as f:
                    lines = f.readlines()
                    all_data[filename] = lines
                    if filename == 'coord.raw':
                        num_configurations = len(lines)
        
        if num_configurations > 0:
            num_to_extract = 100
            indices = list(range(num_configurations)) if num_configurations < num_to_extract else random.sample(range(num_configurations), num_to_extract)
            
            for filename in raw_filenames:
                if filename in all_data:
                    with open(os.path.join(test_path, filename), 'w') as f:
                        for idx in indices:
                            f.write(all_data[filename][idx])
                            
            for extra_file in ['type.raw', 'type_map.raw']:
                if os.path.exists(os.path.join(train_path, extra_file)):
                    shutil.copy(os.path.join(train_path, extra_file), os.path.join(test_path, extra_file))
            
            print(f"  -> Sampled data from root path: {system}")
            
            for path in [train_path, test_path]:
                script_path = os.path.join(path, "raw_to_set.sh")
                with open(script_path, "w") as f:
                    f.write(bash_script_content)
                os.chmod(script_path, 0o755)
                generated_data_paths.append(path)
                
            rel_train = os.path.join("..", "..", "..", os.path.relpath(train_path, current_base_dir))
            rel_test = os.path.join("..", "..", "..", os.path.relpath(test_path, current_base_dir))

            train_sub_content = f'''#!/bin/bash
#SBATCH --job-name={system.lower()}
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --gres=gpu:1
#SBATCH --partition=gpu
#SBATCH --time=24:00:00
#SBATCH --mail-type=end
#SBATCH --mail-user={unum}@smail.iitm.ac.in

python -m deepmd train input.json
'''

            for i in range(graph_num):
                current_seed = seed_values[i] if i < len(seed_values) else seed_values[-1]
                
                base_input_json = {
                    "_comment": " model parameters",
                    "model": {
                        "type_map": nsym_list,
                        "descriptor": {
                            "type": "se_e2_a",
                            "sel": [120, 64], 
                            "rcut_smth": 3.00,
                            "rcut": 6.00,
                            "neuron": [120, 120, 120],
                            "resnet_dt": False,
                            "axis_neuron": 12,
                            "seed": current_seed,
                            "_comment": "that's all"
                        },
                        "fitting_net": {
                            "neuron": [120, 120, 120], 
                            "resnet_dt": True, 
                            "seed": current_seed,
                            "_comment": " that's all"
                        },
                        "_comment": " that's all"
                    },
                    "learning_rate": {
                        "type": "exp", 
                        "start_lr": 0.005, 
                        "decay_steps": 5000,
                        "decay_rate": 0.95,
                        "_comment": "that's all"
                    },
                    "loss": {
                        "type": "ener",
                        "start_pref_e": 0.02, 
                        "limit_pref_e": 8, 
                        "start_pref_f": 1000, 
                        "limit_pref_f": 1, 
                        "start_pref_v": 0.00, 
                        "limit_pref_v": 0,
                        "_comment": " that's all"
                    },
                    "training": {
                        "set_prefix": "set",
                        "stop_batch": 1000000, 
                        "batch_size": 1,
                        "seed": current_seed,
                        "_comment": " display and restart",
                        "disp_file": "lcurve.out", 
                        "disp_freq": 100, 
                        "numb_test": 10, 
                        "save_freq": 1000, 
                        "disp_training": True, 
                        "time_training": True, 
                        "profiling": False, 
                        "profiling_file": "timeline.json",
                        "training_data": {
                            "systems": [rel_train], 
                            "set_prefix": "set", 
                            "batch_size": "auto"
                        },
                        "validation_data": {
                            "systems": [rel_test], 
                            "set_prefix": "set", 
                            "batch_size": "auto"
                        }
                    }
                }
                
                model_seed_path = os.path.join(dp_models_root, system, str(i + 1))
                os.makedirs(model_seed_path, exist_ok=True)
                
                with open(os.path.join(model_seed_path, "input.json"), "w") as f:
                    json.dump(base_input_json, f, indent=4)
                sub_path = os.path.join(model_seed_path, "train.sub")
                with open(sub_path, "w") as f:
                    f.write(train_sub_content)
                os.chmod(sub_path, 0o755)

print("\n--- Automatically Running raw_to_set.sh across all directories ---")
for target_dir in sorted(list(set(generated_data_paths))):
    display_path = os.path.relpath(target_dir, current_base_dir)
    print(f"Running data conversion in: {display_path}...")
    subprocess.run(["./raw_to_set.sh"], cwd=target_dir, shell=True)

print("\nFinished Water setup and all conversion sets are generated!")

Enter your username (for SLURM mail):  ch24d003
Enter the number of DP_models to generate per Water system:  3



--- Processing WATER Systems with Relative Paths ---

Processing Track: DPMD_Water
  -> Sampled data from root path: DPMD_Water

--- Automatically Running raw_to_set.sh across all directories ---
Running data conversion in: Raw_Files/DPMD_Water...
Running data conversion in: Raw_Files/DPMD_Water/Test...

Finished Water setup and all conversion sets are generated!
